# Unit 10.1: Feedforward Neural Networks: Functional Approximation, Backpropagation, and Deep Training Dynamics
Part 10: Deep Learning Foundations

This notebook accompanies Unit 10.1, focusing on:
1. **Universal Functional Approximation**: How multi-layer ReLU networks partition continuous non-linear functions into piecewise linear facets.
2. **From-Scratch Backpropagation**: Exact numerical verification of the 2-hidden-layer network derived by hand in the theoretical notes.
3. **Activation Function Dynamics**: Visualizing gradients, tail saturation, and dying neurons across Sigmoid, Tanh, ReLU, ELU, and Swish.
4. **Regularization Mechanics**: $L_2$ Weight Decay shrinkage, Dropout (training vs. inference modes), and Batch Normalization.
5. **Optimization Dynamics**: Cosine Annealing, Learning Rate Warmup, and batch size gradient noise trade-offs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Reproducibility
np.random.seed(42)

## 1. Universal Functional Approximation: Depth vs. Width
The Universal Approximation Theorem (Cybenko 1989, Hornik 1991) guarantees that a single hidden layer can approximate any continuous function, but Montufar et al. (2014) showed that depth partitions the input space exponentially into $\mathcal{O}((k/d)^{(L-1)d})$ linear facets with ReLU.

In [ ]:
# Target complex continuous 1D physical function
x_dense = np.linspace(-2, 2, 500).reshape(-1, 1)
y_true = np.sin(3 * x_dense) + 0.5 * np.cos(7 * x_dense)

# Train a 2-hidden-layer MLP using Scikit-Learn's MLPRegressor
from sklearn.neural_network import MLPRegressor

mlp = MLPRegressor(hidden_layer_sizes=(32, 32), activation='relu', max_iter=2000, random_state=42)
mlp.fit(x_dense, y_true.ravel())
y_pred = mlp.predict(x_dense)

plt.figure(figsize=(9, 4.5))
plt.plot(x_dense, y_true, 'k-', linewidth=2, label='True Target Function $f^*(x) = \sin(3x) + 0.5\cos(7x)$')
plt.plot(x_dense, y_pred, 'r--', linewidth=2, label='2-Hidden-Layer ReLU MLP Fit (32, 32)')
plt.title('Universal Functional Approximation via Deep ReLU Composition', fontsize=12, fontweight='bold')
plt.xlabel('Input $x$', fontsize=11)
plt.ylabel('Output $y$', fontsize=11)
plt.legend(frameon=True)
plt.grid(True, alpha=0.3)
plt.show()

## 2. From-Scratch Backpropagation: Exact Verification of the 2-Hidden-Layer By-Hand Example
We implement the exact forward and backward pass from Unit 10.1:
- Input: $\mathbf{x} = [0.5, 0.8]^T$, Target: $y = 1.0$, Loss: $\mathcal{L} = \frac{1}{2}(\hat{y} - y)^2$
- Layer 1: $\mathbf{W}^{(1)} = \begin{bmatrix} 0.2 & 0.4 \\ -0.3 & 0.5 \end{bmatrix}, \mathbf{b}^{(1)} = [0.1, -0.2]^T$
- Layer 2: $\mathbf{W}^{(2)} = \begin{bmatrix} 0.6 & -0.4 \\ 0.3 & 0.7 \end{bmatrix}, \mathbf{b}^{(2)} = [0.2, 0.1]^T$
- Layer 3: $\mathbf{W}^{(3)} = [0.5, -0.6], b^{(3)} = 0.3$
- Activation: Sigmoid $\sigma(z) = \frac{1}{1 + e^{-z}}$ throughout.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def d_sigmoid(z):
    s = sigmoid(z)
    return s * (1.0 - s)

# Data
x = np.array([0.5, 0.8])
y = 1.0

# Initialize parameters exactly as in the theoretical notes
W1 = np.array([[0.2, 0.4], [-0.3, 0.5]])
b1 = np.array([0.1, -0.2])
W2 = np.array([[0.6, -0.4], [0.3, 0.7]])
b2 = np.array([0.2, 0.1])
W3 = np.array([0.5, -0.6])
b3 = 0.3

# --- FORWARD PASS ---
z1 = W1 @ x + b1
a1 = sigmoid(z1)

z2 = W2 @ a1 + b2
a2 = sigmoid(z2)

z3 = float(W3 @ a2 + b3)
y_hat = sigmoid(z3)
loss = 0.5 * (y_hat - y) ** 2

print("=== FORWARD PASS VERIFICATION ===")
print(f"Layer 1: z1 = {z1}, a1 = {np.round(a1, 4)}")
print(f"Layer 2: z2 = {np.round(z2, 4)}, a2 = {np.round(a2, 4)}")
print(f"Layer 3: z3 = {z3:.4f}, Prediction y_hat = {y_hat:.4f}")
print(f"Squared Error Loss: {loss:.6f}")

# --- BACKWARD PASS ---
delta3 = (y_hat - y) * d_sigmoid(z3)
grad_W3 = delta3 * a2
grad_b3 = delta3

delta2 = (W3 * delta3) * d_sigmoid(z2)
grad_W2 = np.outer(delta2, a1)
grad_b2 = delta2

delta1 = (W2.T @ delta2) * d_sigmoid(z1)
grad_W1 = np.outer(delta1, x)
grad_b1 = delta1

print("\n=== BACKWARD PASS GRADIENTS ===")
print(f"Output delta3: {delta3:.6f}")
print(f"grad_W3: {np.round(grad_W3, 6)}, grad_b3: {grad_b3:.6f}")
print(f"delta2:  {np.round(delta2, 6)}")
print(f"delta1:  {np.round(delta1, 6)}")

# --- GRADIENT DESCENT UPDATE STEP (eta = 0.5) ---
eta = 0.5
W3_new = W3 - eta * grad_W3
b3_new = b3 - eta * grad_b3
W2_new = W2 - eta * grad_W2
b2_new = b2 - eta * grad_b2
W1_new = W1 - eta * grad_W1
b1_new = b1 - eta * grad_b1

# Re-evaluate forward pass
z1_new = W1_new @ x + b1_new
a1_new = sigmoid(z1_new)
z2_new = W2_new @ a1_new + b2_new
a2_new = sigmoid(z2_new)
z3_new = float(W3_new @ a2_new + b3_new)
y_hat_new = sigmoid(z3_new)
loss_new = 0.5 * (y_hat_new - y) ** 2

print("\n=== UPDATE VERIFICATION ===")
print(f"Old Prediction: {y_hat:.4f}  -->  New Prediction: {y_hat_new:.4f} (Closer to {y})")
print(f"Old Loss:       {loss:.6f}  -->  New Loss:       {loss_new:.6f} (Reduced by {loss - loss_new:.6f})")

## 3. Activation Functions: Gradient Dynamics and Vanishing Gradients
Compare how activation functions behave across their input domain and inspect their local derivatives.
- **Sigmoid / Tanh**: Saturate rapidly outside $[-3, 3]$, driving $\sigma'(z) \to 0$.
- **ReLU**: Constant gradient $1.0$ for $z > 0$, but zero gradient for $z < 0$ (Dying ReLU).
- **Swish / SiLU**: $f(z) = z \cdot \sigma(z)$, smooth non-monotonic curvature.

In [ ]:
z = np.linspace(-4, 4, 400)

# Activations
sig_act = 1.0 / (1.0 + np.exp(-z))
tanh_act = np.tanh(z)
relu_act = np.maximum(0, z)
swish_act = z * (1.0 / (1.0 + np.exp(-z)))

# Derivatives
sig_grad = sig_act * (1.0 - sig_act)
tanh_grad = 1.0 - tanh_act ** 2
relu_grad = (z > 0).astype(float)
swish_grad = swish_act + sig_act * (1.0 - swish_act)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(z, sig_act, label='Sigmoid $\sigma(z)$', linewidth=2)
axes[0].plot(z, tanh_act, label='Tanh $\tanh(z)$', linewidth=2)
axes[0].plot(z, relu_act, label='ReLU $\max(0, z)$', linewidth=2)
axes[0].plot(z, swish_act, label='Swish $z \cdot \sigma(z)$', linewidth=2)
axes[0].set_title('Activation Functions $f(z)$', fontweight='bold')
axes[0].set_xlabel('Pre-activation $z$')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(z, sig_grad, label='Sigmoid Grad (Max 0.25)', linewidth=2)
axes[1].plot(z, tanh_grad, label='Tanh Grad (Max 1.0)', linewidth=2)
axes[1].plot(z, relu_grad, label='ReLU Grad (0 or 1)', linewidth=2)
axes[1].plot(z, swish_grad, label='Swish Grad', linewidth=2)
axes[1].set_title('Activation Derivatives $f'(z)$', fontweight='bold')
axes[1].set_xlabel('Pre-activation $z$')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Learning Rate Schedules: Cosine Annealing vs. Step Decay
Learning rate schedules dynamically balance broad exploration early in training with stable convergence into broad flat minima late in training.

In [ ]:
epochs = 100
t = np.arange(epochs)
lr_max = 0.01
lr_min = 1e-5

# 1. Step Decay (multiplied by 0.1 every 30 epochs)
step_decay = lr_max * (0.1 ** (t // 30))

# 2. Cosine Annealing (Loshchilov & Hutter 2016)
cosine_anneal = lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(t / epochs * np.pi))

# 3. Warmup (first 10 epochs) + Cosine Annealing
warmup_epochs = 10
warmup_lr = np.zeros(epochs)
for ep in range(epochs):
    if ep < warmup_epochs:
        warmup_lr[ep] = lr_max * (ep + 1) / warmup_epochs
    else:
        progress = (ep - warmup_epochs) / (epochs - warmup_epochs)
        warmup_lr[ep] = lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(progress * np.pi))

plt.figure(figsize=(9, 4.5))
plt.plot(t, step_decay, label='Step Decay (Milestones at 30, 60)', linewidth=2)
plt.plot(t, cosine_anneal, label='Cosine Annealing', linewidth=2)
plt.plot(t, warmup_lr, label='10-Epoch Linear Warmup + Cosine', linewidth=2.5, linestyle='--')
plt.title('Learning Rate Schedules and Annealing Strategies', fontsize=12, fontweight='bold')
plt.xlabel('Training Epoch', fontsize=11)
plt.ylabel('Learning Rate $\eta$', fontsize=11)
plt.yscale('log')
plt.grid(True, alpha=0.3, which='both')
plt.legend(frameon=True)
plt.show()

## 5. Weight Initialization Dynamics: Simulating 10-Layer Variance Propagation
We simulate signal propagation through a 10-layer deep network ($n=512$ neurons per layer) with ReLU activations under:
1. **Naive Gaussian** $\mathcal{N}(0, 1)$: Variance explodes exponentially $\mathcal{O}((n\sigma^2)^L)$.
2. **Xavier / Glorot** $\text{Var}(W) = 1/n_{\text{in}}$: Fails under ReLU because zeroing out negative inputs cuts variance in half at each layer ($2^{-L}$), vanishing to zero.
3. **He / Kaiming** $\text{Var}(W) = 2/n_{\text{in}}$: Exactly compensates for the ReLU halving, preserving constant variance across all 10 layers.

In [ ]:
layers = 10
width = 512
n_samples = 1000

# Input data with unit variance
X_init = np.random.randn(n_samples, width)

def trace_variance(init_fn, activation_fn):
    a = X_init.copy()
    variances = [np.var(a)]
    for l in range(layers):
        W = init_fn(width, width)
        z = a @ W
        a = activation_fn(z)
        variances.append(np.var(a))
    return variances

# 1. Naive (sigma = 0.1)
var_naive = trace_variance(lambda n_in, n_out: np.random.randn(n_in, n_out) * 0.1, lambda z: np.maximum(0, z))

# 2. Xavier with ReLU
var_xavier = trace_variance(lambda n_in, n_out: np.random.randn(n_in, n_out) * np.sqrt(1.0 / n_in), lambda z: np.maximum(0, z))

# 3. He / Kaiming with ReLU
var_kaiming = trace_variance(lambda n_in, n_out: np.random.randn(n_in, n_out) * np.sqrt(2.0 / n_in), lambda z: np.maximum(0, z))

layer_indices = np.arange(layers + 1)

plt.figure(figsize=(9, 4.5))
plt.plot(layer_indices, var_naive, 'r-o', label='Naive Gaussian ($\\sigma=0.1$)', linewidth=2)
plt.plot(layer_indices, var_xavier, 'b--s', label='Xavier / Glorot (Vanishes by $2^{-L}$ on ReLU)', linewidth=2)
plt.plot(layer_indices, var_kaiming, 'g-^', label='He / Kaiming (Stable Unit Variance on ReLU)', linewidth=2.5)
plt.title('Layer-by-Layer Activation Variance Across 10 Deep Layers (ReLU)', fontsize=12, fontweight='bold')
plt.xlabel('Layer Depth $l$', fontsize=11)
plt.ylabel('Activation Variance $\\mathrm{Var}(a^{(l)})$', fontsize=11)
plt.yscale('log')
plt.grid(True, alpha=0.3, which='both')
plt.legend(frameon=True)
plt.show()

print("Layer 10 Final Variance:")
print(f"  Naive:   {var_naive[-1]:.6e}")
print(f"  Xavier:  {var_xavier[-1]:.6e} (Vanished by factor of ~{var_xavier[0]/var_xavier[-1]:.0f})")
print(f"  Kaiming: {var_kaiming[-1]:.4f} (Maintained near 1.0)")